# Ejecución de inferencia y harness

## 1. Clonar repositorio

In [18]:
# Clonar el repo (codigo versionado, incluye config.yaml y fuentes.yaml)
!git clone --branch refactor-harness --single-branch https://github.com/luisNP21/Topicos-IA.git /content/corpus

fatal: destination path '/content/corpus' already exists and is not an empty directory.


In [19]:
%cd /content/corpus
%ls

/content/corpus
M1/  M2/  M3/  M4/  README.md  review/


## 2. Instalar dependencias

In [20]:
!pip install -r M2/run_inference/requirements.txt
!pip install -r M2/harness/requirements.txt

  Using cached peft-0.20.0-py3-none-any.whl.metadata (14 kB)
Using cached peft-0.20.0-py3-none-any.whl (775 kB)
  Attempting uninstall: peft
    Found existing installation: peft 0.21.2
    Uninstalling peft-0.21.2:
      Successfully uninstalled peft-0.21.2
  Attempting uninstall: python-dotenv
    Found existing installation: python-dotenv 1.2.4
    Uninstalling python-dotenv-1.2.4:
      Successfully uninstalled python-dotenv-1.2.4
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2/2 [groq]


## 3. Configuración de paths y checks de recursos necesitados

In [21]:
from google.colab import drive
drive.mount("/content/drive")

from pathlib import Path
import shutil
import yaml

PROJECT_ROOT = Path("/content/drive/MyDrive/TopicosIA/Proyecto-Salud")

# 1. Detectar o asegurar la clonación del repositorio (rama LuisM4)
candidatos_repo = [
    Path("/content/corpus"),
    Path("/content/Topicos-IA"),
    Path.cwd(),
]
REPO = next((c for c in candidatos_repo if (c / "M2/run_inference/config.yaml").exists()), None)
if REPO is None:
    !git clone --branch LuisM4 --single-branch https://github.com/luisNP21/Topicos-IA.git /content/corpus
    REPO = Path("/content/corpus")

# 2. Asegurar el Gold Set de los 15 casos (texto clínico original) en Drive
gold_dest = PROJECT_ROOT / "M2/eval_harness/gold_casos.jsonl"
if not gold_dest.exists():
    candidatos_gold = [
        PROJECT_ROOT / "M2/eval_harness/gold_m4_limpio.jsonl",
        REPO / "M4/eval_set/eval_set_casos.jsonl",
    ]
    for c in candidatos_gold:
        if c.exists():
            gold_dest.parent.mkdir(parents=True, exist_ok=True)
            shutil.copy2(c, gold_dest)
            break

checks = {
    "Adaptador LoRA": PROJECT_ROOT / "M1/saved_models/clinical_bert-distemist-lora/adapter_model.safetensors",
    "Gold set": gold_dest,
}
for nombre, path in checks.items():
    print(f"{nombre}: {'OK' if path.exists() else 'FALTA'} -- {path}")

# 3. Configurar el harness normal (modo texto, sin visión) sobre los 15 casos
gold_rel = str(gold_dest.relative_to(PROJECT_ROOT)).replace("\\", "/")

# Config de inferencia
ri_cfg_path = REPO / "M2/run_inference/config.yaml"
with open(ri_cfg_path, "r", encoding="utf-8") as f:
    ri_cfg = yaml.safe_load(f)
ri_cfg["eval_set_path"] = gold_rel
ri_cfg["fuente"] = {"tipo": "texto"}  # Texto directo original, sin visión
ri_cfg["output_path"] = "M2/predictions/encoder_solo.json"
with open(ri_cfg_path, "w", encoding="utf-8") as f:
    yaml.safe_dump(ri_cfg, f, sort_keys=False)

# Config del harness
h_cfg_path = REPO / "M2/harness/config.yaml"
with open(h_cfg_path, "r", encoding="utf-8") as f:
    h_cfg = yaml.safe_load(f)
h_cfg["rutas"]["gold_set"] = gold_rel
h_cfg["rutas"]["outputs_dir"] = "M2/outputs"
with open(h_cfg_path, "w", encoding="utf-8") as f:
    yaml.safe_dump(h_cfg, f, sort_keys=False)

print(f"\nRepo ubicado en: {REPO}")
print(f"Configs actualizados: eval_set_path = {gold_rel}")

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
Adaptador LoRA: OK -- /content/drive/MyDrive/TopicosIA/Proyecto-Salud/M1/saved_models/clinical_bert-distemist-lora/adapter_model.safetensors
Gold set: OK -- /content/drive/MyDrive/TopicosIA/Proyecto-Salud/M2/eval_harness/gold_casos.jsonl

Repo ubicado en: /content/corpus
Configs actualizados: eval_set_path = M2/eval_harness/gold_casos.jsonl


In [22]:
from pathlib import Path
from google.colab import userdata

with open(REPO / ".env", "w") as f:
    f.write(f"PROJECT_ROOT={PROJECT_ROOT}\n")
    f.write(f"GROQ_API_KEY={userdata.get('GROQ_API_KEY')}\n")
print(f".env generado en {REPO / '.env'}")

.env generado en /content/corpus/.env


## 4. Correr inferencia

In [23]:
%cd M2/run_inference/

/content/corpus/M2/run_inference


In [24]:
!python run_inference.py

Skipping import of cpp extensions due to incompatible torch version 2.11.0+cpu for torchao version 0.16.0             Please see https://github.com/pytorch/ao/issues/2919 for more info
config.json: 100% 613/613 [00:00<00:00, 2.32MB/s]

pytorch_model.bin: downloading bytes:  26% 131M/504M [00:01<00:02, 169MB/s, 10.3MB/s  ]
pytorch_model.bin: downloading bytes:  34% 171M/504M [00:01<00:01, 179MB/s, 14.3MB/s  ]
pytorch_model.bin: downloading bytes:  50% 250M/504M [00:01<00:01, 160MB/s, 21.7MB/s  ]
pytorch_model.bin: downloading bytes:  64% 322M/504M [00:02<00:01, 177MB/s, 26.9MB/s  ]
pytorch_model.bin: reconstructing file:  80% 402M/504M [00:02<00:00, 206MB/s, 24.8MB/s  ]
pytorch_model.bin: downloading bytes: 100% 335M/335M [00:04<00:00, 69.8MB/s, 29.2MB/s  ]
pytorch_model.bin: reconstructing file: 100% 504M/504M [00:04<00:00, 105MB/s, 38.8MB/s  ] 
Loading weights: 100% 197/197 [00:00<00:00, 12725.67it/s]
[transformers] RobertaForTokenClassification LOAD REPORT from: PlanTL-GOB-ES/roberta

## 5. Correr harness

In [25]:
%cd ..

/content/corpus/M2


In [26]:
%cd harness
%ls

/content/corpus/M2/harness
cached_system.py    harness.py              metrics_semantic.py  scorecard.py
config_vision.yaml  length_bias_pairs.json  __pycache__/
config.yaml         metrics_exact.py        requirements.txt
gold_loader.py      metrics_judge.py        run_harness.py


In [27]:
!python run_harness.py

Proyecto en: /content/drive/MyDrive/TopicosIA/Proyecto-Salud
Gold set: /content/drive/MyDrive/TopicosIA/Proyecto-Salud/M2/eval_harness/gold_casos.jsonl | existe: True
Predicciones: /content/drive/MyDrive/TopicosIA/Proyecto-Salud/M2/predictions/encoder_solo.json | existe: True
Output: /content/drive/MyDrive/TopicosIA/Proyecto-Salud/M2/outputs
Gold set cargado: 15 ejemplos
Formato valido
Primer ejemplo:
  text[:120]: Varón de 52 años, con sobrepeso (IMC 29.3 kg/m2) y vida sedentaria, que acude a consulta de medicina preventiva. En anal ...
  entities_gold: ['sobrepeso', 'enfermedad cardiovascular', 'diabetes mellitus tipo 2']
  n_entidades: 3
[04:49:50] Seed global fijada: 42
[04:49:50] PROJECT_ROOT: /content/drive/MyDrive/TopicosIA/Proyecto-Salud
[04:49:50] DIMENSION 1 -- Exact-match
Proyecto en: /content/drive/MyDrive/TopicosIA/Proyecto-Salud
Gold set: /content/drive/MyDrive/TopicosIA/Proyecto-Salud/M2/eval_harness/gold_casos.jsonl | existe: True
Predicciones: /content/drive/MyDrive/To